In [1]:
import lfox
import lfox.lattice as lat
import jax
import jax.numpy as jnp

import numpy as np

In [2]:
np.ones(4)

array([1., 1., 1., 1.])

In [3]:
d = 2
L = 4

In [4]:
MyLat = lat.SquareLattice(dims=((L,)*d))
MyLat

In [15]:
phi_field = lat.LatticeField(MyLat)
phi_field.field = np.ones_like(phi_field.field)
phi_field.field[0,0] = 0.
phi_field.field = jnp.array(phi_field.field)
print(phi_field.field, phi_field)

[[0. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]] <lfox.lattice.LatticeField object at 0x122a5b0d0>


In [6]:
phi_field.nn((3,0), 0)

([0, 0], 1.0)

In [7]:
@jax.jit
def action_lfox(F):
    S = 0.0
    phi = F.field
    for i in range(L):
        for j in range(L):
            s = (i,j)
            for ax in range(d):
                s_nn, bc_mul = F.nn(s, ax)
#                S -= 2 * phi_field.field[s] * phi_field.nn(s, ax)
                S -= 2 * phi[s] * phi[s_nn] * bc_mul

    phi2 = phi_field.field**2
    S += jnp.sum(phi2)
    S += jnp.sum( (phi2-1)**2)

    return S

In [8]:
Z = np.array(np.meshgrid(np.arange(2), np.arange(2), np.arange(2))).T.reshape(-1,3)
print(Z.shape, Z)
I = np.ones((2,2,2))
print(I.shape, I[Z[0]].shape)
np.array([ I[tuple(Zi)]  for Zi in Z ])

(8, 3) [[0 0 0]
 [0 1 0]
 [1 0 0]
 [1 1 0]
 [0 0 1]
 [0 1 1]
 [1 0 1]
 [1 1 1]]
(2, 2, 2) (3, 2, 2)


array([1., 1., 1., 1., 1., 1., 1., 1.])

In [9]:
@jax.jit
def action(phi):
    S = 0.0
    for ax in range(d):
        S -= 2*jnp.sum(phi * jnp.roll(phi, shift=1, axis=ax))
    phi2 = phi**2
    S += jnp.sum(phi2)
    S += jnp.sum( (phi2-1)**2 )
    return S

@jax.jit
def act2(phi):
    S = 0.0
    for i in range(L):
        for j in range(L):
            i_nn = (i+1) % L
            j_nn = (j+1) % L
            S -= 2 * phi[i,j] * phi[i_nn,j]
            S -= 2 * phi[i,j] * phi[i,j_nn]
    phi2 = phi**2
    S += jnp.sum(phi2)
    S += jnp.sum( (phi2-1)**2 )
    return S

@jax.jit
def act3(phi):
    S = 0.0
    all_sites = np.array(np.meshgrid(*[np.arange(L) for _ in range(d)]).T.reshape(-1,d))

        

In [13]:
phi_ones = np.ones((16,)*d)
%time action(phi_ones)

%time act2(phi_ones)

CPU times: user 679 µs, sys: 365 µs, total: 1.04 ms
Wall time: 696 µs
CPU times: user 233 µs, sys: 133 µs, total: 366 µs
Wall time: 252 µs


Array(192., dtype=float32)

In [14]:
%time action_lfox(phi_field)

AttributeError: DynamicJaxprTracer has no attribute field

In [71]:
phi_rand = np.random.rand(16,16)


In [73]:
print(action(phi_rand))
print(act2(phi_rand))

-70.96844
-70.96853
